# Boosting: o que cada botão faz

**Para quem está começando.** Este notebook supõe que você já viu o notebook de bagging e boosting, da mesma pasta. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. ver a troca entre taxa de aprendizado e número de árvores;
2. medir o que a profundidade faz com o treino e com a validação;
3. medir o efeito de amostrar linhas e colunas dentro do boosting;
4. usar parada antecipada para não decidir o número de árvores no chute.

Roda no Google Colab, e a célula abaixo instala os pacotes de boosting.


## Os botões, um por um

O boosting tem mais botões que os outros modelos, e cada um tem um efeito diferente. Vale saber o que esperar de cada um antes de girar:

- **taxa de aprendizado** (`learning_rate`): quanto cada árvore corrige do erro que sobrou. Baixa significa passos curtos e mais árvores; alta chega rápido e tem mais chance de passar do ponto;
- **número de árvores** (`n_estimators`): o tamanho do orçamento de correções. Sozinho não diz nada: o que conta é o produto dele com a taxa;
- **profundidade** (`max_depth`): quanto cada árvore pode interagir entre colunas. Em tabela, boosting costuma usar árvores rasas, de 3 a 6 níveis, porque quem corrige o erro são as próximas;
- **amostragem** (`subsample` e `colsample_bytree`): cada árvore vê uma parte das linhas e das colunas. É bagging dentro do boosting, e costuma ajudar em dado com ruído;
- **regularização** (`reg_lambda`, `reg_alpha`): o mesmo papel do Ridge e do Lasso do notebook de regressão, agora no tamanho das folhas;
- **parada antecipada**: em vez de escolher o número de árvores no chute, guarde uma parte do treino, acompanhe a métrica e pare quando ela parar de melhorar. É o botão mais importante na prática.


## Passo 0: ambiente e configuração

A semente é a mesma em todas as células, e a divisão de treino e teste é a do resto do repositório.


In [ ]:
%pip install -q xgboost==3.4.1 lightgbm==4.7.0 catboost==1.2.10

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (9, 4)

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=RANDOM_STATE,
)
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print(f"✅ {X_train.shape[0]} linhas de treino | {X_test.shape[0]} de teste | semente {RANDOM_STATE}")

## Passo 1: taxa de aprendizado contra número de árvores

A grade abaixo cruza cinco taxas com quatro tamanhos de orçamento, medindo a acurácia de validação cruzada em cada combinação. A leitura é mais fácil no mapa de calor do que na tabela: quanto mais escuro, melhor.


In [ ]:
taxas = [0.01, 0.05, 0.1, 0.3, 1.0]
arvores = [10, 50, 100, 300]

linhas = []
for taxa in taxas:
    for n in arvores:
        modelo = xgb.XGBClassifier(n_estimators=n, learning_rate=taxa, max_depth=4,
                                   random_state=RANDOM_STATE)
        acuracia = cross_val_score(modelo, X_train, y_train, cv=folds, scoring="accuracy")
        linhas.append({"taxa": taxa, "arvores": n, "acuracia_cv": acuracia.mean()})

grade = pd.DataFrame(linhas).pivot(index="taxa", columns="arvores", values="acuracia_cv")
display(grade.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
imagem = ax.imshow(grade.values, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(grade.columns)), grade.columns)
ax.set_yticks(range(len(grade.index)), grade.index)
ax.set_xlabel("número de árvores")
ax.set_ylabel("taxa de aprendizado")
ax.set_title("Acurácia de validação cruzada por combinação")
for i in range(grade.shape[0]):
    for j in range(grade.shape[1]):
        ax.text(j, i, f"{grade.values[i, j]:.3f}", ha="center", va="center", fontsize=8)
fig.colorbar(imagem, ax=ax)
plt.tight_layout()
plt.show()

### Como ler isso

A diagonal escura é a lição: taxa pequena pede muitas árvores, taxa grande chega ao mesmo lugar com poucas. Existe um orçamento comum por trás disso, e ele é aproximadamente o produto entre a taxa e o número de árvores.

Dois cantos são ruins por motivos diferentes. Taxa baixa com poucas árvores deixa o modelo longe de convergir: ele começou a aprender e parou no meio. Taxa alta com poucas árvores chega rápido ao platô, mas a um platô mais baixo: com passos grandes, o ajuste fino deixa de ser possível.

Na prática, usa-se taxa baixa com muitas árvores, e o número de árvores vem da parada antecipada, não desta grade. A grade serve para o que ela fez aqui: mostrar que existe uma superfície, e não um valor mágico.


## Passo 2: profundidade, e o preço da capacidade

A profundidade controla quantas interações entre colunas cada árvore pode representar. Árvore de profundidade 1 só olha uma coluna por vez; de 8, pode combinar oito colunas numa regra. A linha com amostragem e regularização entra ao lado para mostrar o outro lado do ajuste.


In [ ]:
linhas = []
for profundidade in range(1, 9):
    modelo = xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=profundidade,
                               random_state=RANDOM_STATE)
    modelo.fit(X_train, y_train)
    acuracia = cross_val_score(modelo, X_train, y_train, cv=folds, scoring="accuracy")
    linhas.append({"max_depth": profundidade, "acuracia_cv": acuracia.mean(),
                   "desvio": acuracia.std(), "acuracia_treino": modelo.score(X_train, y_train)})

ajustado = xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=6, subsample=0.7,
                            colsample_bytree=0.7, reg_lambda=5.0, random_state=RANDOM_STATE)
ajustado.fit(X_train, y_train)
linhas.append({"max_depth": "6 + amostragem e reg.", "acuracia_cv": cross_val_score(
    ajustado, X_train, y_train, cv=folds, scoring="accuracy").mean(),
    "desvio": np.nan, "acuracia_treino": ajustado.score(X_train, y_train)})

tabela_profundidade = pd.DataFrame(linhas).set_index("max_depth")
display(tabela_profundidade.round(4))

### Como ler isso

Olhe as duas colunas de acurácia conforme a profundidade cresce. A profundidade 1 é a única que erra no treino: uma árvore de um nível olha uma coluna por vez e não representa interação nenhuma. Da profundidade 2 para cima, a acurácia de treino já está no teto, o modelo decorou o treino, e a de validação não acompanha: ela até cai um pouco.

Esse afastamento entre as duas colunas é o overfitting, medido, e a lição é dura: com boosting, quase toda a capacidade extra vira memorização, e é por isso que em tabela se usa árvore rasa. O ganho de mexer na profundidade é pequeno, e o custo aparece em dado com ruído.

A última linha mostra o outro caminho: mesma capacidade com menos liberdade. A amostragem de linhas e colunas e a regularização reduziram a memorização do treino sem perder acurácia de validação, e em dado com ruído ou com poucas linhas isso vale mais que girar a profundidade.

Duas coisas para experimentar: leve a profundidade para 10 e veja se a validação cai de verdade ou só empata; e troque `subsample=0.7` por `0.5`, para ver se o resultado fica mais estável entre folds (olhe o `desvio`).


## Passo 3: parada antecipada, o botão que evita o chute

A ideia é simples: em vez de decidir o número de árvores antes, separe uma parte do treino só para acompanhar a métrica e pare quando ela parar de melhorar. O modelo devolve em qual iteração isso aconteceu.

O detalhe que importa: essa parte de acompanhamento sai **de dentro do treino**, e o teste continua intocado até o fim. Usar o teste para decidir quando parar é a forma mais fácil de publicar um número otimista.


In [ ]:
X_ajuste, X_validacao, y_ajuste, y_validacao = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE,
)

ORCAMENTO = 1000
antecipado = xgb.XGBClassifier(n_estimators=ORCAMENTO, learning_rate=0.1, max_depth=4,
                               early_stopping_rounds=20, random_state=RANDOM_STATE)
inicio = time.perf_counter()
antecipado.fit(X_ajuste, y_ajuste, eval_set=[(X_validacao, y_validacao)], verbose=False)
tempo_antecipado = time.perf_counter() - inicio

sem_parada = xgb.XGBClassifier(n_estimators=ORCAMENTO, learning_rate=0.1, max_depth=4,
                               random_state=RANDOM_STATE)
inicio = time.perf_counter()
sem_parada.fit(X_train, y_train)
tempo_sem_parada = time.perf_counter() - inicio

tabela_parada = pd.DataFrame([
    {"modelo": f"com parada (parou em {antecipado.best_iteration} de {ORCAMENTO})",
     "acuracia_teste": antecipado.score(X_test, y_test), "treino_s": tempo_antecipado},
    {"modelo": f"sem parada ({ORCAMENTO} árvores)", "acuracia_teste": sem_parada.score(X_test, y_test),
     "treino_s": tempo_sem_parada},
]).set_index("modelo")
display(tabela_parada.round(4))

### Como ler isso

A parada antecipada fez três coisas de uma vez: escolheu o número de árvores com dado, gastou menos tempo de treino e usou um pedaço do treino para vigiar em vez de usar o teste. É o padrão em produção, e é o motivo pelo qual "quantas árvores?" deixou de ser uma pergunta importante nessas bibliotecas.

Vale registrar a assimetria do experimento: o modelo com parada viu 80% do treino, e o outro viu 100%. Mesmo com menos dado, ele foi melhor no teste, e o teste não participou de nenhuma decisão.

O `early_stopping_rounds=20` é o quanto o modelo espera antes de desistir: se a métrica de acompanhamento não melhora em 20 árvores seguidas, ele para e guarda a melhor iteração, não a última. Com ruído maior, esse número sobe; com dado limpo, desce.

## Resumo

Boosting é uma sequência de árvores rasas, cada uma corrigindo o erro que sobrou. Os botões se dividem em três grupos: o orçamento (taxa de aprendizado vezes número de árvores), a capacidade de cada árvore (profundidade), e o controle de liberdade (amostragem e regularização). A parada antecipada resolve o primeiro grupo com dado, em vez de com chute.

O erro mais comum em quem começa: girar a profundidade procurando acurácia. Em tabela, a profundidade alta só aumenta a acurácia de treino; quem melhora a validação é o conjunto certo de taxa, orçamento e liberdade, medido com validação cruzada e com desvio ao lado.

**Próximo passo:** os mesmos botões numa tarefa de regressão, onde a métrica vira erro em vez de acurácia, e a parada antecipada acompanha RMSE em vez de log-loss.

### Referências

**Os botões e o ajuste**

Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* The Annals of Statistics 29(5), 1189-1232.

Chen, T., & Guestrin, C. (2016). *XGBoost: a scalable tree boosting system.* Proceedings of KDD.

**Ajuste de hiperparâmetros**

Bergstra, J., & Bengio, Y. (2012). *Random search for hyper-parameter optimization.* Journal of Machine Learning Research 13, 281-305.

```bibtex
@article{friedman2001greedy,
  title   = {Greedy function approximation: a gradient boosting machine},
  author  = {Friedman, Jerome H.},
  journal = {The Annals of Statistics},
  volume  = {29},
  number  = {5},
  pages   = {1189--1232},
  year    = {2001}
}

@inproceedings{chen2016xgboost,
  title     = {XGBoost: a scalable tree boosting system},
  author    = {Chen, Tianqi and Guestrin, Carlos},
  booktitle = {Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining},
  year      = {2016}
}

@article{bergstra2012random,
  title   = {Random search for hyper-parameter optimization},
  author  = {Bergstra, James and Bengio, Yoshua},
  journal = {Journal of Machine Learning Research},
  volume  = {13},
  pages   = {281--305},
  year    = {2012}
}
```
